# Dataset Generation

Generates all synthetic target function datasets used in the paper:
- `datasets_1d_jaderberg.pkl` — Ω₁ = {1, 1.2, 3} (replicates Jaderberg et al. 2024)
- `datasets_1d_jaderberg_10.pkl` — Ω₂ = {11, 11.2, 13} (shifted by +10)
- `datasets_1d_jaderberg_GaussianShift.pkl` — Ω₁ + Gaussian offset sweep

Run this notebook once before any experiment notebooks.
Output `.pkl` files are saved to the `datasets/` directory (configurable below).

In [1]:
import numpy as np
import jax.numpy as jnp
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path
import pickle

SEED = 42
np.random.seed(SEED)

# ── Output directory ───────────────────────────────────────────
OUT_DIR = Path("../datasets")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Parameters

In [2]:
NUM_FUNCTIONS  = 10
NUM_SAMPLES    = 100   # x-values per function
TEST_PCTG      = 0.20

# Target frequency sets
FREQS_JAD    = np.array([1.0,  1.2,  3.0])   # Ω₁ — Jaderberg original
FREQS_JAD_10 = np.array([11.0, 11.2, 13.0])  # Ω₂ — shifted by +10

# Gaussian offset sweep parameters (for robustness figure)
MU_VALUES     = np.linspace(0, 9, 10)         # mean offsets
SIGMA_OFFSET  = 1.0                            # std of per-frequency noise

## Helper functions

In [3]:
def target_function(freqs, coeff0, coeffs_a, coeffs_b, x):
    """Truncated Fourier series with cosine + sine terms."""
    res = coeff0
    for i in range(len(freqs)):
        res += coeffs_a[i] * np.cos(freqs[i] * x) + \
               coeffs_b[i] * np.sin(freqs[i] * x)
    return res


def make_dataset(freqs, num_functions, num_samples, test_pctg, seed):
    """Generate a list of train/test dicts for a given frequency set."""
    rng = np.random.RandomState(seed)
    x_vals = np.linspace(-np.pi, np.pi, num_samples)
    scaler_x = MinMaxScaler(feature_range=(-np.pi, np.pi))
    scaler_y = MinMaxScaler(feature_range=(-1, 1))
    x_scaled = jnp.array(
        scaler_x.fit_transform(x_vals.reshape(-1, 1))
    )

    datasets = []
    for _ in range(num_functions):
        c0  = rng.random()
        ca  = rng.random(len(freqs))
        cb  = rng.random(len(freqs))
        y   = np.array([target_function(freqs, c0, ca, cb, xv)
                        for xv in x_vals])
        y_s = jnp.array(scaler_y.fit_transform(y.reshape(-1, 1)))
        x_tr, x_te, y_tr, y_te = train_test_split(
            x_scaled, y_s, test_size=test_pctg, random_state=seed
        )
        datasets.append({
            "x_all": x_scaled, "y_all": y_s,
            "x_train": x_tr,   "y_train": y_tr,
            "x_test":  x_te,   "y_test":  y_te,
            "freqs": freqs, "coeff0": c0, "coeffs_a": ca, "coeffs_b": cb,
        })
    return datasets


def make_gaussian_shift_datasets(base_freqs, mu_values, sigma,
                                  num_functions, num_samples,
                                  test_pctg, seed):
    """For each mu, generate datasets with freqs = base_freqs + N(mu, sigma)."""
    all_shifts = {}
    for mu in mu_values:
        rng = np.random.RandomState(seed)
        shifted_freqs = base_freqs + rng.normal(mu, sigma, size=len(base_freqs))
        shifted_freqs = np.abs(shifted_freqs)  # keep positive
        all_shifts[float(mu)] = make_dataset(
            shifted_freqs, num_functions, num_samples, test_pctg, seed
        )
    return all_shifts

## Generate and save

In [4]:
print("Generating Ω₁ datasets (Jaderberg original)...")
ds_jad = make_dataset(FREQS_JAD, NUM_FUNCTIONS, NUM_SAMPLES, TEST_PCTG, SEED)
with open(OUT_DIR / "datasets_1d_jaderberg.pkl", "wb") as f:
    pickle.dump(ds_jad, f)
print(f"  Saved {len(ds_jad)} functions to datasets_1d_jaderberg.pkl")

print("Generating Ω₂ datasets (shifted +10)...")
ds_jad_10 = make_dataset(FREQS_JAD_10, NUM_FUNCTIONS, NUM_SAMPLES, TEST_PCTG, SEED)
with open(OUT_DIR / "datasets_1d_jaderberg_10.pkl", "wb") as f:
    pickle.dump(ds_jad_10, f)
print(f"  Saved {len(ds_jad_10)} functions to datasets_1d_jaderberg_10.pkl")

print("Generating Gaussian offset sweep datasets...")
ds_gauss = make_gaussian_shift_datasets(
    FREQS_JAD, MU_VALUES, SIGMA_OFFSET,
    NUM_FUNCTIONS, NUM_SAMPLES, TEST_PCTG, SEED
)
with open(OUT_DIR / "datasets_1d_jaderberg_GaussianShift.pkl", "wb") as f:
    pickle.dump(ds_gauss, f)
print(f"  Saved {len(ds_gauss)} offset levels to datasets_1d_jaderberg_GaussianShift.pkl")

print("\nDone. All datasets saved to", OUT_DIR)

Generating Ω₁ datasets (Jaderberg original)...
  Saved 10 functions to datasets_1d_jaderberg.pkl
Generating Ω₂ datasets (shifted +10)...
  Saved 10 functions to datasets_1d_jaderberg_10.pkl
Generating Gaussian offset sweep datasets...
  Saved 10 offset levels to datasets_1d_jaderberg_GaussianShift.pkl

Done. All datasets saved to ../datasets
